# CIFAR-10 Experiment Replication from Di et al. (2025)
### *Computationally Efficient General Convolutional Layer Selection for Transfer Learning*
**Paper section:** Section 4.1: CIFAR-10 Experiment.

---
### 📌 Reproducible effects:
1. **Two transfer modes:**
   - **Imbalance Case:** Non-overlapping classes (Source = 5 animal classes, Target = 5 vehicle classes).
   - **Balance Case:** Split of 10 classes (50% source objects, 50% target).
2. **Reproduction of Figure 4 (A, B):** Generality measure profile $\omega_j$ with confidence intervals for sample sizes $n \in \{500, 1000, 1500\}$.
3. **Reproduction of Figure 4 (C, D):** Dependence of accuracy and error on the number of transferred layers $M_0 \dots M_6$.
4. **Computational complexity comparison:** Algorithm 1 speedup estimate compared with classical exhaustive search $M_0 \dots M_6$.

## 1. Google Colab Environment Setup and GPU Connection

In [ ]:
import os
import sys
import time
import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Mounting Google Drive when running in Colab
if 'google.colab' in sys.modules:
    print("[Colab] Google Colab environment detected.")
    from google.colab import drive
    drive.mount('/content/drive')
    
    candidates = [
        '/content/drive/MyDrive/user/goit/Diplom/var1_Di/code/Di',
        '/content/drive/MyDrive/Diplom/var1_Di/code/Di',
        '/content/drive/MyDrive/var1_Di/code/Di',
        '/content/drive/MyDrive/code/Di'
    ]
    target_dir = None
    for cand in candidates:
        if os.path.isdir(cand):
            target_dir = cand
            break
            
    if target_dir is None:
        print("[Colab] Searching for project folder in MyDrive...")
        for root, dirs, files in os.walk('/content/drive/MyDrive'):
            if 'cifar10_experiment.py' in files and 'di_algorithm.py' in files:
                target_dir = root
                break
                
    if target_dir:
        os.chdir(target_dir)
        if target_dir not in sys.path:
            sys.path.insert(0, target_dir)
        print(f"[Colab] Working directory: {os.getcwd()}")
else:
    current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
    if current_dir not in sys.path:
        sys.path.insert(0, current_dir)
    print(f"[Local] Directory: {os.getcwd()}")

# GPU Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Experiment Parameters and Mode Selection

In [ ]:
# Interactive experiment parameters
EXPERIMENT_CASE = 'imbalance'   # 'imbalance' (animals -> vehicles) or 'balance' (50% / 50%)
SAMPLE_SIZES = [500, 1000]      # Sample sizes for PC profiling (in the paper: 500, 1000, 1500)
NUM_REPLICATIONS = 2            # Number of repetitions for mean and std calculation
PRETRAIN_EPOCHS = 8             # Pretraining epochs of the base CNN on source
TARGET_EPOCHS = 8               # Adaptation epochs on the target task
USE_REAL_CIFAR10 = False        # True: load official CIFAR-10; False: fast synthetic mode

print(f"Experiment mode: {EXPERIMENT_CASE}")
print(f"PC sample sizes: {SAMPLE_SIZES}")
print(f"Using real CIFAR-10: {USE_REAL_CIFAR10}")

## 3. Data Loading and Source / Target Split Creation

In [ ]:
from models import CIFAR6CNN, create_transferred_model
from dataset_splits import (
    split_cifar_imbalance,
    split_cifar_balance,
    generate_synthetic_transfer_dataset,
    make_dataloaders,
    CIFAR_SOURCE_CLASSES,
    CIFAR_TARGET_CLASSES
)
from di_algorithm import (
    compute_pc_profile,
    estimate_phase_transition,
    forward_adding_layer_selection,
    train_target_candidate,
    default_eval_loss
)
from cifar10_experiment import load_cifar10_raw

if USE_REAL_CIFAR10:
    print("Loading full CIFAR-10 dataset (60,000 images)...")
    images, labels = load_cifar10_raw(data_dir='./data')
    if EXPERIMENT_CASE == 'imbalance':
        splits = split_cifar_imbalance(images, labels)
    else:
        splits = split_cifar_balance(images, labels)
else:
    print("Generating fast synthetic benchmark for Colab demonstration...")
    splits = generate_synthetic_transfer_dataset(
        n_source=1200, n_target=800,
        num_classes=5, image_size=32,
        transition_layer=3 if EXPERIMENT_CASE == 'imbalance' else 4,
        seed=42
    )

loaders = make_dataloaders(splits, batch_size=64)
print(f"Source: training={len(splits['source_train'][0])}, test={len(splits['source_test'][0])}")
print(f"Target:   training={len(splits['target_train'][0])}, test={len(splits['target_test'][0])}")

## 4. Pretraining 6-Layer CNN on Base Dataset (Source)

In [ ]:
source_model = CIFAR6CNN(num_classes=splits['num_source_classes']).to(device)
optimizer = torch.optim.RMSprop(source_model.parameters(), lr=0.002, alpha=0.9)
criterion = nn.CrossEntropyLoss()

print(f"Pretraining on Source ({PRETRAIN_EPOCHS} epochs)...")
source_model.train()
for ep in range(1, PRETRAIN_EPOCHS + 1):
    tr_loss = 0.0
    for x, y in loaders['source_train']:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = source_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * x.size(0)
    avg_loss = tr_loss / len(loaders['source_train'].dataset)
    v_loss, v_acc = default_eval_loss(source_model, loaders['source_test'], criterion, device)
    if ep == 1 or ep % 2 == 0 or ep == PRETRAIN_EPOCHS:
        print(f"  Epoch {ep:02d}/{PRETRAIN_EPOCHS:02d} | Train Loss: {avg_loss:.4f} | Source Test Acc: {v_acc * 100:.1f}%")

## 5. Computing Generality Profiles $\omega_j$ (Reproduction of Figure 4A / 4B)

In the Di et al. (2025) paper, the profile $\omega_j = \mathrm{PC}(\Phi^j(X^t), Y^t)$ is evaluated for different sample sizes $n \in \{500, 1000, 1500\}$ with multiple repetitions.

In [ ]:
pc_results = {}
layer_names = CIFAR6CNN.LAYER_NAMES

for n_s in SAMPLE_SIZES:
    print(f"\nPC estimation for sample size n = {n_s} ({NUM_REPLICATIONS} repetitions)...")
    pc_results[n_s] = {l: [] for l in layer_names}
    for rep in range(NUM_REPLICATIONS):
        prof = compute_pc_profile(
            source_model, loaders['target_train'],
            device=device, max_samples=n_s,
            verbose=False
        )
        for l, val in prof['omega'].items():
            pc_results[n_s][l].append(val)
            
    for l in layer_names:
        mean_v = np.mean(pc_results[n_s][l])
        std_v = np.std(pc_results[n_s][l])
        print(f"  {l:<8}: mean = {mean_v:.4f} +/- {std_v:.4f}")

# Determining the theoretical optimum k*
last_n = SAMPLE_SIZES[-1]
mean_profile = {l: float(np.mean(pc_results[last_n][l])) for l in layer_names}
best_layer = max(mean_profile, key=mean_profile.get)
k_star_theory = layer_names.index(best_layer) + 1
print(f"\nTheoretical cut-point (argmax omega_j): k* = {k_star_theory} ({best_layer})")

## 6. Figure 4 Plot (A/B): Layer Generality Measure with Errors

In [ ]:
plt.figure(figsize=(10, 5))
markers = ['o-', 's--', '^:']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
layer_idx = np.arange(1, len(layer_names) + 1)

for idx, n_s in enumerate(SAMPLE_SIZES):
    means = [np.mean(pc_results[n_s][l]) for l in layer_names]
    stds = [np.std(pc_results[n_s][l]) for l in layer_names]
    plt.errorbar(
        layer_idx, means, yerr=stds,
        fmt=markers[idx % len(markers)], color=colors[idx % len(colors)],
        capsize=4, linewidth=2, markersize=7,
        label=f'n = {n_s}'
    )

# Theoretical boundary line
plt.axvline(x=k_star_theory, color='crimson', linestyle='--', linewidth=2,
            label=f'Theoretical cut-point $k^* = {k_star_theory}$ ({best_layer})')

plt.xticks(layer_idx, [f"L{i}: {l}" for i, l in zip(layer_idx, layer_names)], fontsize=10)
plt.xlabel('Convolutional Layers', fontsize=12, fontweight='bold')
plt.ylabel('Generality Measure $\omega_j$ (PC)', fontsize=12, fontweight='bold')
plt.title(f'Figure 4: Layer Generality Profile ({EXPERIMENT_CASE.capitalize()} Case)', fontsize=13, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Running Algorithm 1: Forward-Adding with Warm-Start

In [ ]:
t0_algo = time.time()
algo_result = forward_adding_layer_selection(
    source_model,
    target_train_loader=loaders['target_train'],
    target_val_loader=loaders['target_test'],
    num_target_classes=splits['num_target_classes'],
    pc_sample_size=SAMPLE_SIZES[0],
    train_epochs=TARGET_EPOCHS,
    lr=0.001,
    freeze_transferred=True,
    device=device,
    verbose=True
)
algo_time = time.time() - t0_algo
print(f"\nAlgorithm 1 execution time: {algo_time:.2f} sec.")

## 8. Exhaustive Search of Models $M_0 \dots M_6$ (Reproduction of Figure 4C / 4D)

In [ ]:
print("Training complete set of nested models M_0 ... M_6...")
exhaustive_results = []
t0_ex = time.time()

for k in range(0, 7):
    print(f"  Training model M_{k} ({k} transferred layers)...")
    _, v_loss, v_acc = train_target_candidate(
        source_model, k=k,
        num_target_classes=splits['num_target_classes'],
        train_loader=loaders['target_train'],
        val_loader=loaders['target_test'],
        epochs=TARGET_EPOCHS, lr=0.001,
        freeze_transferred=True,
        device=device, verbose=False
    )
    exhaustive_results.append({'k': k, 'val_loss': v_loss, 'val_acc': v_acc})
    print(f"    M_{k}: Val Loss = {v_loss:.4f} | Val Acc = {v_acc*100:.1f}%")

exhaustive_time = time.time() - t0_ex
best_exhaustive = min(exhaustive_results, key=lambda x: x['val_loss'])

## 9. Figure 4 Plot (C/D) and Comparative Analysis

In [ ]:
k_list = [r['k'] for r in exhaustive_results]
losses = [r['val_loss'] for r in exhaustive_results]
accs = [r['val_acc'] * 100 for r in exhaustive_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot (Figure 4C)
ax1.plot(k_list, accs, 'o-', color='#1f77b4', linewidth=2.5, markersize=8)
ax1.axvline(x=best_exhaustive['k'], color='crimson', linestyle='--', label=f'Optimum $k^*={best_exhaustive["k"]}$')
ax1.set_xlabel('Number of Transferred Layers $k$', fontsize=11, fontweight='bold')
ax1.set_ylabel('Validation Accuracy (%)', fontsize=11, fontweight='bold')
ax1.set_title('Figure 4C: Validation Accuracy of Models $M_k$', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend()

# Loss function plot (Figure 4D)
ax2.plot(k_list, losses, 's-', color='#d62728', linewidth=2.5, markersize=8)
ax2.axvline(x=best_exhaustive['k'], color='crimson', linestyle='--', label=f'Optimum $k^*={best_exhaustive["k"]}$')
ax2.set_xlabel('Number of Transferred Layers $k$', fontsize=11, fontweight='bold')
ax2.set_ylabel('Validation Loss', fontsize=11, fontweight='bold')
ax2.set_title('Figure 4D: Validation Error of Models $M_k$', fontsize=12, fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

# Summary report
print("\n" + "=" * 70)
print(f"FINAL COMPARISON (Di et al., 2025 - {EXPERIMENT_CASE.upper()} CASE)")
print("=" * 70)
print(f"Theoretical calculation (argmax omega_j): k* = {k_star_theory}")
print(f"Algorithm 1 (Forward-Adding):         k* = {algo_result['selected_k']} | Time: {algo_time:.2f} sec.")
print(f"Exhaustive Search:  k* = {best_exhaustive['k']} | Time: {exhaustive_time:.2f} sec.")
print(f"Actual algorithm speedup:     {exhaustive_time / max(0.001, algo_time):.1f}x faster!")
print("=" * 70)

## 10. Saving Results to `results/`

In [ ]:
os.makedirs('results', exist_ok=True)
out_file = f"results/colab_{EXPERIMENT_CASE}_replication.json"

with open(out_file, 'w', encoding='utf-8') as f:
    json.dump({
        'experiment_case': EXPERIMENT_CASE,
        'k_star_theory': k_star_theory,
        'algo_selected_k': algo_result['selected_k'],
        'exhaustive_best_k': best_exhaustive['k'],
        'algo_time_sec': algo_time,
        'exhaustive_time_sec': exhaustive_time,
        'speedup': exhaustive_time / max(0.001, algo_time),
        'exhaustive_results': exhaustive_results
    }, f, indent=2)

print(f"Experiment results successfully saved to: {out_file}")